# Retos del workshop RAG · SI3003

Notebook de retos de Juan David Sierra Orozco. Parte de `04_challenges_starter.ipynb` y completa los TODO.

Retos resueltos aquí, en este orden: **C1** (HNSW contra fuerza bruta), **C4** (evaluación automática hit@k), **C3** (app Gradio con filtro por paper), **C5** (MMR) y **C6** (persistencia en disco). Las respuestas escritas están en `RESPUESTAS.md`.

## C1 · HNSW contra fuerza bruta con FAISS

Esta parte solo usa `numpy` y `faiss`, no necesita ninguna API. Decisiones de medición:

- La verdad exacta sale de `IndexFlatL2`. El recall@10 de HNSW se mide contra esos 10 vecinos.
- La latencia es de **una consulta a la vez y con un solo hilo** (`omp_set_num_threads(1)`), que es lo que vive un servicio en línea y evita que el paralelismo de FAISS favorezca a uno u otro índice. Se toma el mejor de 3 pasadas.
- La construcción usa todos los hilos de la máquina (solo se reporta como referencia).

In [ ]:
%pip install -q faiss-cpu matplotlib pandas

In [ ]:
import os
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EVID = Path("../evidencias") if Path("../evidencias").exists() else Path("evidencias")
EVID.mkdir(exist_ok=True)

N_HILOS = os.cpu_count() or 1
d, n, nq, k = 64, 20_000, 200, 10
rng = np.random.default_rng(42)
xb = rng.random((n, d), dtype="float32")    # base de vectores
xq = rng.random((nq, d), dtype="float32")   # consultas

def recall_at_k(I, I_true):
    return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(I, I_true)]))

def ms_por_consulta(index, consultas, k, pasadas=3):
    """Latencia de una consulta a la vez, un hilo, mejor de varias pasadas."""
    faiss.omp_set_num_threads(1)
    mejor = float("inf")
    for _ in range(pasadas):
        t0 = time.perf_counter()
        for i in range(len(consultas)):
            index.search(consultas[i:i + 1], k)
        mejor = min(mejor, (time.perf_counter() - t0) / len(consultas) * 1000)
    faiss.omp_set_num_threads(N_HILOS)
    return mejor

def tabla_md(df, decimales=3):
    cols = list(df.columns)
    filas = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in df.iterrows():
        celdas = [f"{v:.{decimales}f}" if isinstance(v, float) else str(v) for v in r]
        filas.append("| " + " | ".join(celdas) + " |")
    return "\n".join(filas)

print(f"faiss {faiss.__version__}, {N_HILOS} hilos disponibles, d={d}, n={n}, nq={nq}, k={k}")

### Línea base: exacto frente a HNSW con n = 20.000

Código de arranque de la guía, con la medición de latencia por consulta. `efSearch` siempre es mayor o igual que `k = 10`.

In [ ]:
def construir_exacto(base):
    flat = faiss.IndexFlatL2(base.shape[1])
    flat.add(base)
    return flat

def construir_hnsw(base, M, ef_construction=100):
    index = faiss.IndexHNSWFlat(base.shape[1], M)
    index.hnsw.efConstruction = ef_construction
    faiss.omp_set_num_threads(N_HILOS)
    t0 = time.perf_counter()
    index.add(base)
    return index, time.perf_counter() - t0

flat = construir_exacto(xb)
_, I_true = flat.search(xq, k)
t_flat = ms_por_consulta(flat, xq, k)
print(f"exacto (IndexFlatL2): {t_flat:.3f} ms/consulta")

EFS = (10, 16, 32, 64, 128, 256)
MS = (8, 16, 32)
filas = []
indices_hnsw = {}
for M in MS:
    index, build_s = construir_hnsw(xb, M)
    indices_hnsw[M] = index
    for ef in EFS:
        index.hnsw.efSearch = ef
        _, I = index.search(xq, k)
        filas.append(dict(M=M, efSearch=ef, recall10=recall_at_k(I, I_true),
                          ms_consulta=ms_por_consulta(index, xq, k), build_s=build_s))
df = pd.DataFrame(filas)
df["vs_exacto"] = t_flat / df["ms_consulta"]   # >1: HNSW más rápido que el exacto
print(df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

### Gráficas: recall frente a efSearch y recall frente a latencia

In [ ]:
colores = {8: "#4c78a8", 16: "#e45756", 32: "#54a24b"}

# Figura 1: recall@10 contra efSearch, una curva por M
fig, ax = plt.subplots(figsize=(7, 4.5))
for M, g in df.groupby("M"):
    ax.plot(g["efSearch"], g["recall10"], marker="o", color=colores[M], label=f"M={M}")
ax.axhline(0.95, color="gray", linestyle="--", linewidth=1, label="recall = 0,95")
ax.set_xscale("log", base=2)
ax.set_xticks(EFS)
ax.set_xticklabels(EFS)
ax.set_xlabel("efSearch")
ax.set_ylabel("recall@10 frente al exacto")
ax.set_title(f"HNSW sobre vectores aleatorios (n={n:,}, d={d})".replace(",", "."))
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(EVID / "C1_recall_efsearch.png", dpi=150)
plt.show()

# Figura 2: recall@10 contra latencia, con la latencia del exacto como referencia
fig, ax = plt.subplots(figsize=(7, 4.5))
for M, g in df.groupby("M"):
    ax.plot(g["ms_consulta"], g["recall10"], marker="o", color=colores[M], label=f"M={M}")
    for _, r in g.iterrows():
        ax.annotate(int(r["efSearch"]), (r["ms_consulta"], r["recall10"]),
                    textcoords="offset points", xytext=(4, -10), fontsize=7, color=colores[M])
ax.axvline(t_flat, color="black", linestyle="--", linewidth=1, label=f"exacto ({t_flat:.2f} ms, recall = 1)")
ax.axhline(0.95, color="gray", linestyle=":", linewidth=1)
ax.set_xlabel("milisegundos por consulta (un hilo)")
ax.set_ylabel("recall@10 frente al exacto")
ax.set_title("Recall contra latencia (los números son efSearch)")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(EVID / "C1_recall_latencia.png", dpi=150)
plt.show()

### Pregunta 1: qué efSearch da recall ≥ 0,95 con M = 16

In [ ]:
g16 = df[df.M == 16].sort_values("efSearch")
ok = g16[g16.recall10 >= 0.95]
if len(ok):
    ef95 = int(ok.iloc[0]["efSearch"])
    print(f"Con M=16, el menor efSearch de la rejilla con recall >= 0,95 es {ef95} "
          f"(recall {ok.iloc[0]['recall10']:.3f}, {ok.iloc[0]['ms_consulta']:.3f} ms/consulta "
          f"frente a {t_flat:.3f} ms del exacto).")
else:
    print("Ningún efSearch de la rejilla llega a 0,95 con M=16. Máximo:", g16.recall10.max())
# refinamiento: barrer efSearch más fino en el índice M=16
index16 = indices_hnsw[16]
fino = []
for ef in (10, 16, 24, 32, 48, 64, 96, 128, 160, 192, 256, 384, 512):
    index16.hnsw.efSearch = ef
    _, I = index16.search(xq, k)
    fino.append((ef, recall_at_k(I, I_true)))
print("Barrido fino con M=16:", ", ".join(f"ef={e}: {r:.3f}" for e, r in fino))

### Pregunta 2: desde qué n HNSW supera al exacto

Barrido con n = 20.000, 100.000 y 500.000 (efConstruction = 100, mismas 200 consultas, M = 16 y también M = 32). 500.000 × 64 × 4 bytes son unos 128 MB de datos más el grafo, así que cabe de sobra en los 16 GB del Mac y se prueba completo, sin reducir n.

Comparar a un efSearch fijo engaña: HNSW con efSearch bajo es muy rápido pero devuelve resultados malos. Por eso, además de la rejilla completa, busco para cada n **el menor efSearch que alcanza recall ≥ 0,90 y ≥ 0,95** (hasta efSearch = 4096) y comparo la latencia en ese punto contra el exacto.

In [ ]:
EF_AMPLIO = (16, 32, 64, 128, 256, 512, 1024, 2048, 4096)
filas_n = []
for n_i in (20_000, 100_000, 500_000):
    rng_i = np.random.default_rng(7)
    xb_i = rng_i.random((n_i, d), dtype="float32")
    flat_i = construir_exacto(xb_i)
    faiss.omp_set_num_threads(N_HILOS)
    _, I_true_i = flat_i.search(xq, k)
    t_flat_i = ms_por_consulta(flat_i, xq, k)
    for M_i in (16, 32):
        idx_i, build_i = construir_hnsw(xb_i, M_i)
        for ef in EF_AMPLIO:
            idx_i.hnsw.efSearch = ef
            _, I = idx_i.search(xq, k)
            rec = recall_at_k(I, I_true_i)
            ms = ms_por_consulta(idx_i, xq, k, pasadas=2)
            filas_n.append(dict(n=n_i, M=M_i, efSearch=ef, recall10=rec, ms_hnsw=ms,
                                ms_exacto=t_flat_i, aceleracion=t_flat_i / ms, build_s=build_i))
            if rec >= 0.999:
                break
        del idx_i
    del flat_i, xb_i
df_n = pd.DataFrame(filas_n)
print("Rejilla completa con M=16:")
print(df_n[df_n.M == 16].to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
resumen_n = []
for (n_i, M_i), g in df_n.groupby(["n", "M"]):
    for objetivo in (0.90, 0.95):
        ok = g[g.recall10 >= objetivo].sort_values("efSearch")
        if len(ok):
            r = ok.iloc[0]
            resumen_n.append(dict(n=n_i, M=M_i, recall_objetivo=objetivo, efSearch=int(r.efSearch), recall10=r.recall10,
                                  ms_hnsw=r.ms_hnsw, ms_exacto=r.ms_exacto, aceleracion=r.aceleracion))
        else:
            resumen_n.append(dict(n=n_i, M=M_i, recall_objetivo=objetivo, efSearch=-1, recall10=g.recall10.max(),
                                  ms_hnsw=float("nan"), ms_exacto=g.ms_exacto.iloc[0], aceleracion=float("nan")))
df_resumen_n = pd.DataFrame(resumen_n)
print("Menor efSearch que alcanza el recall objetivo (aceleracion > 1 significa que HNSW gana al exacto):")
print(df_resumen_n.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for M_i, ax in zip((16, 32), axes):
    for ef, g in df_n[(df_n.M == M_i) & df_n.efSearch.isin([32, 64, 128, 256])].groupby("efSearch"):
        ax.plot(g["n"], g["aceleracion"], marker="o", label=f"efSearch={ef}")
    ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("n (vectores en la base)")
    ax.set_ylabel("aceleración = ms exacto / ms HNSW")
    ax.set_title(f"M={M_i}: cuándo HNSW le gana a la fuerza bruta")
    ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(EVID / "C1_aceleracion_vs_n.png", dpi=150)
plt.show()

### Por qué los datos aleatorios son un caso difícil: contraste con datos agrupados

Los puntos uniformes en 64 dimensiones casi no tienen estructura: las distancias a todos los puntos se concentran (el vecino más cercano está casi tan lejos como el promedio). Para ver el efecto, se repite la rejilla de M = 16 con una mezcla de 100 gaussianas, que imita mejor la estructura de agrupamiento de embeddings reales (aunque sigue siendo un modelo sintético, no son embeddings de texto).

In [ ]:
def mezcla(n_pts, d, n_cl=100, sigma=0.05, semilla=0):
    r = np.random.default_rng(semilla)
    centros = r.random((n_cl, d), dtype="float32")
    asignacion = r.integers(0, n_cl, n_pts)
    return (centros[asignacion] + sigma * r.standard_normal((n_pts, d))).astype("float32"), centros

xb_c, centros = mezcla(n, d)
r2 = np.random.default_rng(1)
xq_c = (centros[r2.integers(0, 100, nq)] + 0.05 * r2.standard_normal((nq, d))).astype("float32")
flat_c = construir_exacto(xb_c)
_, I_true_c = flat_c.search(xq_c, k)
t_flat_c = ms_por_consulta(flat_c, xq_c, k)
idx_c, _ = construir_hnsw(xb_c, 16)
filas_c = []
for ef in EFS:
    idx_c.hnsw.efSearch = ef
    _, I = idx_c.search(xq_c, k)
    filas_c.append(dict(efSearch=ef, recall10_agrupado=recall_at_k(I, I_true_c),
                        recall10_aleatorio=float(df[(df.M == 16) & (df.efSearch == ef)].recall10.iloc[0])))
df_c = pd.DataFrame(filas_c)
print(df_c.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# Concentración de distancias: contraste relativo entre vecino más cercano y distancia media
def contraste(base, consultas):
    D = ((consultas[:50, None, :] - base[None, :5000, :]) ** 2).sum(-1) ** 0.5
    return float(np.mean(D.min(1) / D.mean(1)))
print(f"distancia al vecino más cercano / distancia media: aleatorio = {contraste(xb, xq):.3f}, agrupado = {contraste(xb_c, xq_c):.3f}")

## Preparación común para C4, C3, C5 y C6

Reconstruyo aquí el pipeline del notebook 03 (MarkItDown, `RecursiveCharacterTextSplitter`, `NVIDIAEmbeddings`, Qdrant). La clave `NVIDIA_API_KEY` se lee del `.env` del repositorio (o de los secretos de Colab) y nunca se imprime.

La clave se comparte con otros procesos, así que todas las llamadas a la API van **en serie, con reintentos y espera exponencial** ante errores 429 y 5xx, y la indexación se hace por lotes pequeños con una pausa entre lotes.

In [ ]:
%pip install -q "markitdown[pdf]" langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv requests gradio playwright

In [ ]:
import os
import random
import getpass
import time
import warnings
from pathlib import Path
from typing import List

import requests
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore", message=".*type is unknown and inference may fail.*")

# find_dotenv(usecwd=True) busca .env en notebooks/ y sube por los directorios padre (raíz del repo).
load_dotenv(find_dotenv(usecwd=True))

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY (.env del repo o secretos de Colab).")
os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY

# CAMBIO DE MODELO: el notebook 03 usa "nvidia/nemotron-3-super-120b-a12b", pero mi cuenta de NVIDIA NO lo
# tiene habilitado (responde 404). Uso "nvidia/nemotron-3.5-super-vl-120b-a12b", que sí está disponible.
# Es un modelo con razonamiento: gasta tokens "pensando" antes de responder, por eso subo
# max_completion_tokens a 2500 y, si aun así la respuesta sale cortada (finish_reason = length), se reintenta con 6000.
GENERATION_MODEL = "nvidia/nemotron-3.5-super-vl-120b-a12b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
print("Configuración NVIDIA NIM lista (la clave no se muestra).")

In [ ]:
from langchain_core.embeddings import Embeddings
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA

TRANSITORIOS = ("429", "500", "502", "503", "504", "timed out", "Timeout", "Connection", "temporarily")

def con_reintentos(fn, *args, intentos=7, base=2.0, **kwargs):
    """Llama a fn; ante 429/5xx o fallos de red espera base*2^i s (con jitter) y reintenta."""
    for i in range(intentos):
        try:
            return fn(*args, **kwargs)
        except Exception as e:  # noqa: BLE001
            if i == intentos - 1 or not any(t in str(e) for t in TRANSITORIOS):
                raise
            time.sleep(base * 2 ** i + random.random())

class EmbeddingsRobustos(Embeddings):
    """NVIDIAEmbeddings con reintentos y caché de consultas (la misma pregunta se embebe una vez)."""
    def __init__(self, base):
        self.base = base
        self.cache_consultas = {}
        self.llamadas_consulta = 0
    def embed_query(self, text):
        if text not in self.cache_consultas:
            self.llamadas_consulta += 1
            self.cache_consultas[text] = con_reintentos(self.base.embed_query, text)
        return self.cache_consultas[text]
    def embed_documents(self, texts):
        return con_reintentos(self.base.embed_documents, list(texts))

embeddings = EmbeddingsRobustos(NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END"))
EMBEDDING_DIM = len(embeddings.embed_query("dimension probe"))
print("Dimensión de embeddings:", EMBEDDING_DIM)

In [ ]:
from markitdown import MarkItDown

DOCS_DIR = Path("../papers") if Path("../papers").exists() else Path("papers")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
}
TITULOS = {
    "attention_paper.pdf": "Attention Is All You Need",
    "rag_paper.pdf": "Retrieval-Augmented Generation",
}

def download_file(url: str, destination: Path) -> None:
    if destination.exists() and destination.stat().st_size > 10_000:
        print(f"Ya existe: {destination.name}")
        return
    response = requests.get(url, timeout=90)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF"):
        raise ValueError(f"La descarga de {url} no produjo un PDF válido")
    destination.write_bytes(response.content)

for filename, url in PAPERS.items():
    download_file(url, DOCS_DIR / filename)

md_converter = MarkItDown(enable_plugins=False)
TEXTOS = {}
for filename in PAPERS:
    texto = md_converter.convert(str(DOCS_DIR / filename)).text_content
    if not texto or not texto.strip():
        raise ValueError(f"No se pudo extraer texto de {filename}")
    TEXTOS[filename] = texto
    print(f"{filename}: {len(texto):,} caracteres")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def pdf_to_documents(text: str, title: str, source: str, chunk_size=900, chunk_overlap=120) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "],
    )
    documents = []
    for index, chunk in enumerate(splitter.split_text(text)):
        if len(chunk.strip()) < 30:
            continue
        documents.append(Document(
            page_content=chunk,
            metadata={"title": title, "source": source, "chunk_index": index},
        ))
    return documents

def documentos_para(chunk_size: int) -> List[Document]:
    """Overlap proporcional (~13 %) para que los tres tamaños sean comparables."""
    overlap = round(chunk_size * 0.13)
    docs = []
    for source, texto in TEXTOS.items():
        docs += pdf_to_documents(texto, TITULOS[source], source, chunk_size, overlap)
    return docs

for cs in (400, 900, 1500):
    print(f"chunk_size={cs}: {len(documentos_para(cs))} chunks")

In [ ]:
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

def indexar_por_lotes(vectorstore, docs, lote=32, pausa=0.3):
    """Embebe e inserta en lotes pequeños, en serie, con reintentos y una pausa entre lotes."""
    for i in range(0, len(docs), lote):
        con_reintentos(vectorstore.add_documents, docs[i:i + lote])
        time.sleep(pausa)

def crear_vectorstore(client, coleccion, docs):
    client.create_collection(
        collection_name=coleccion,
        vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
    )
    vs = QdrantVectorStore(client=client, collection_name=coleccion, embedding=embeddings)
    indexar_por_lotes(vs, docs)
    return vs

# Un solo cliente en memoria para C4, C3 y C5 (C6 usa uno en disco más abajo).
qdrant_client = QdrantClient(":memory:")
print("Cliente Qdrant en memoria listo")

## C4 · Evaluación automática de la recuperación

`eval/preguntas.csv` tiene 10 preguntas (6 sobre el paper de Attention y 4 sobre el de RAG, más la pareja que ya traía la plantilla). Cada fila trae `fuente_esperada` y una `palabra_clave` que debe aparecer en el chunk correcto. **Verifiqué cada palabra clave leyendo el texto extraído de los PDFs** (aparecen literalmente en el texto de MarkItDown, que en algunos pasajes pierde los espacios, por eso uso claves como `labelsmoothing` o `warmup_steps=4000`).

`hit_at_k(retriever, filas)` es la función de la guía: una pregunta cuenta como acierto si algún chunk recuperado contiene la palabra clave **y** viene de la fuente esperada.

In [ ]:
import csv
import pandas as pd

def cargar_preguntas(ruta="../eval/preguntas.csv"):
    if not Path(ruta).exists():
        ruta = "eval/preguntas.csv"
    with open(ruta, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def hit_at_k(retriever, filas):
    hits = 0
    for r in filas:
        docs = retriever.invoke(r["pregunta"])
        ok = any(
            r["palabra_clave"].lower() in d.page_content.lower()
            and d.metadata["source"] == r["fuente_esperada"]
            for d in docs
        )
        hits += ok
    return hits / len(filas)

def hit_por_pregunta(retriever, filas):
    """Misma lógica que hit_at_k pero devuelve 0/1 por pregunta, para ver dónde falla."""
    salida = []
    for r in filas:
        docs = retriever.invoke(r["pregunta"])
        salida.append(int(any(
            r["palabra_clave"].lower() in d.page_content.lower()
            and d.metadata["source"] == r["fuente_esperada"] for d in docs)))
    return salida

filas_eval = cargar_preguntas()
print(len(filas_eval), "preguntas")
for r in filas_eval:
    # verificación del ground truth contra el texto completo del paper esperado
    presente = r["palabra_clave"].lower() in TEXTOS[r["fuente_esperada"]].lower()
    print(f"{r['id']:>2} | {r['fuente_esperada']:<20} | clave {r['palabra_clave']!r:<22} presente en el PDF: {presente}")

In [ ]:
resultados_c4 = []
detalle_c4 = {}
vectorstores = {}
for cs in (400, 900, 1500):
    docs = documentos_para(cs)
    t0 = time.perf_counter()
    vs = crear_vectorstore(qdrant_client, f"papers_cs{cs}", docs)
    t_idx = time.perf_counter() - t0
    vectorstores[cs] = vs
    fila = dict(chunk_size=cs, chunks=len(docs), indexacion_s=round(t_idx, 1))
    for k_ in (3, 5):
        retr = vs.as_retriever(search_kwargs={"k": k_})
        fila[f"hit@{k_}"] = hit_at_k(retr, filas_eval)
        detalle_c4[(cs, k_)] = hit_por_pregunta(retr, filas_eval)
    resultados_c4.append(fila)
    print(fila)
df_c4 = pd.DataFrame(resultados_c4)
df_c4

In [ ]:
print(tabla_md(df_c4.assign(**{"hit@3": df_c4["hit@3"].map(lambda v: f"{v:.0%}"), "hit@5": df_c4["hit@5"].map(lambda v: f"{v:.0%}")})))
# detalle por pregunta: 1 = acierto
det = pd.DataFrame({f"cs{cs}@{k_}": v for (cs, k_), v in detalle_c4.items()},
                   index=[f"P{r['id']} ({r['palabra_clave']})" for r in filas_eval])
det

In [ ]:
# ¿En qué posición del ranking aparece el primer chunk correcto? (top 30, para entender los fallos)
def posicion_primer_acierto(vs, r, tope=30):
    docs = vs.as_retriever(search_kwargs={"k": tope}).invoke(r["pregunta"])
    for pos, d in enumerate(docs, 1):
        if r["palabra_clave"].lower() in d.page_content.lower() and d.metadata["source"] == r["fuente_esperada"]:
            return pos
    return None

rangos = pd.DataFrame({f"cs{cs}": [posicion_primer_acierto(vectorstores[cs], r) for r in filas_eval] for cs in (400, 900, 1500)},
                      index=[f"P{r['id']} ({r['palabra_clave']})" for r in filas_eval])
print("Posición (1 = primero) del primer chunk correcto; None = fuera del top 30")
rangos

## C5 · MMR frente a similitud pura

Con el índice de `chunk_size=900`, comparo `similarity` (k = 5) con `mmr` (`k=5, fetch_k=20, lambda_mult=0.5`) en 3 preguntas. Para medir la redundancia calculo la **similitud coseno media entre los 5 chunks recuperados** (más alta = más redundantes) y cuento cuántos pasajes distintos del documento cubren.

In [ ]:
import numpy as np

vs900 = vectorstores[900]
retr_sim = vs900.as_retriever(search_type="similarity", search_kwargs={"k": 5})
retr_mmr = vs900.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5})

def redundancia(docs):
    E = np.array(embeddings.embed_documents([d.page_content for d in docs]))
    E = E / np.linalg.norm(E, axis=1, keepdims=True)
    S = E @ E.T
    iu = np.triu_indices(len(docs), 1)
    return float(S[iu].mean())

def resumen(docs, n=110):
    return [f"[{d.metadata['source'][:4]} #{d.metadata['chunk_index']:>3}] " + d.page_content.replace("\n", " ")[:n] for d in docs]

preguntas_c5 = [
    "How does the Transformer use attention?",                       # amplia, muchos chunks parecidos
    "How many warmup steps are used in the learning rate schedule?", # puntual, una respuesta
    "What are the retriever and the generator in RAG and how are they trained?",  # compuesta
]
comparacion_c5 = []
for q in preguntas_c5:
    d_sim = con_reintentos(retr_sim.invoke, q)
    d_mmr = con_reintentos(retr_mmr.invoke, q)
    r_sim, r_mmr = redundancia(d_sim), redundancia(d_mmr)
    comparacion_c5.append(dict(
        pregunta=q,
        redundancia_sim=r_sim, redundancia_mmr=r_mmr,
        chunks_distintos_sim=len({d.metadata["chunk_index"] for d in d_sim}),
        chunks_distintos_mmr=len({d.metadata["chunk_index"] for d in d_mmr}),
        coincidentes=len({(d.metadata["source"], d.metadata["chunk_index"]) for d in d_sim}
                         & {(d.metadata["source"], d.metadata["chunk_index"]) for d in d_mmr}),
    ))
    print("=" * 100)
    print("PREGUNTA:", q)
    print(f"\n  SIMILITUD  (redundancia media {r_sim:.3f})")
    for linea in resumen(d_sim): print("   ", linea)
    print(f"\n  MMR        (redundancia media {r_mmr:.3f})")
    for linea in resumen(d_mmr): print("   ", linea)
df_c5 = pd.DataFrame(comparacion_c5)
df_c5

In [ ]:
# ¿Pierde MMR el chunk con la respuesta? Aciertos por palabra clave con las 10 preguntas de C4.
hit_sim = hit_at_k(retr_sim, filas_eval)
hit_mmr = hit_at_k(retr_mmr, filas_eval)
print(f"hit@5 con 10 preguntas (chunk_size=900): similitud = {hit_sim:.0%}, MMR = {hit_mmr:.0%}")

## C3 · App Gradio para los papers con filtro por paper

Interfaz que une el corpus del 03 con la UI del 02b: respuesta, fuentes (título, archivo, score) y un `gr.Dropdown` («Todos», «Attention Is All You Need», «Retrieval-Augmented Generation»). El filtro es un `models.Filter` de Qdrant sobre `metadata.source` (langchain-qdrant guarda los metadatos bajo la clave `metadata`).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

FUENTES = {
    "Todos": None,
    "Attention Is All You Need": "attention_paper.pdf",
    "Retrieval-Augmented Generation": "rag_paper.pdf",
}
vectorstore = vectorstores[900]
COLLECTION_APP = "papers_cs900"

# Comprobación del payload (útil si el filtro no devolviera nada)
punto, _ = qdrant_client.scroll(COLLECTION_APP, limit=1)
print("Payload guardado:", {k_: (v if k_ != "page_content" else v[:60] + "...") for k_, v in punto[0].payload.items()})

def filtro_por_fuente(source):
    if not source or source == "Todos":
        return None
    return models.Filter(must=[
        models.FieldCondition(key="metadata.source", match=models.MatchValue(value=FUENTES.get(source, source)))
    ])

def retriever_for(source, k=5, threshold=0.25):
    kwargs = {"k": k, "score_threshold": threshold}
    flt = filtro_por_fuente(source)
    if flt is not None:
        kwargs["filter"] = flt
    return vectorstore.as_retriever(search_type="similarity_score_threshold", search_kwargs=kwargs)

def recuperar_con_score(pregunta, paper="Todos", k=5, threshold=0.25):
    return con_reintentos(vectorstore.similarity_search_with_score, pregunta, k=k,
                          score_threshold=threshold, filter=filtro_por_fuente(paper))

llm = ChatNVIDIA(model=GENERATION_MODEL, temperature=0.2, top_p=0.95, max_completion_tokens=2500)
llm_largo = ChatNVIDIA(model=GENERATION_MODEL, temperature=0.2, top_p=0.95, max_completion_tokens=6000)

def format_docs(documents):
    return "\n\n---\n\n".join(
        f"[{d.metadata.get('title', 'Untitled')} | {d.metadata.get('source', '')}]\n{d.page_content}"
        for d in documents)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer using only the supplied academic-paper excerpts. "
     "If the excerpts do not contain enough evidence, say so explicitly. "
     "Answer in the same language as the question and be concise. "
     "Mention the paper title supporting the answer."),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])

class CadenaRespuesta:
    """prompt | llm con control de truncamiento: el modelo razona antes de responder y puede quedarse sin tokens."""
    def invoke(self, entrada):
        for modelo in (llm, llm_largo):
            msg = (prompt | modelo).invoke(entrada)
            if msg.content.strip() and msg.response_metadata.get("finish_reason") != "length":
                break
        return msg.content

cadena_respuesta = CadenaRespuesta()

def responder(pregunta, paper="Todos"):
    """Devuelve (respuesta, tabla de fuentes) con el filtro de paper aplicado en Qdrant."""
    if not pregunta or not pregunta.strip():
        return "Escribe una pregunta.", []
    resultados = recuperar_con_score(pregunta, paper)
    if not resultados:
        return "No encontré fragmentos relevantes con ese filtro.", []
    docs = [d for d, _ in resultados]
    respuesta = con_reintentos(cadena_respuesta.invoke, {"context": format_docs(docs), "question": pregunta})
    if not respuesta.strip():  # modelo con razonamiento: se quedó sin tokens visibles
        respuesta = "(El modelo no devolvió texto visible; reintenta.)"
    fuentes = [[d.metadata["title"], d.metadata["source"], round(float(s), 3)] for d, s in resultados]
    return respuesta.strip(), fuentes

# Comparación de una misma pregunta con y sin filtro
PREGUNTA_COMPARA = "How is the model trained?"   # ambigua a propósito: los dos papers entrenan modelos
comparacion_c3 = {}
for etiqueta in ("Todos", "Attention Is All You Need", "Retrieval-Augmented Generation"):
    resp, fuentes = responder(PREGUNTA_COMPARA, etiqueta)
    comparacion_c3[etiqueta] = (resp, fuentes)
    print("=" * 100)
    print(f"FILTRO: {etiqueta}")
    print("Fuentes:", [(f[1], f[2]) for f in fuentes])
    print("Respuesta:", resp)

In [ ]:
import gradio as gr

def ui_responder(pregunta, paper):
    return responder(pregunta, paper)

with gr.Blocks(title="RAG de papers con filtro por paper") as demo:
    gr.Markdown("## RAG de papers académicos\nPregunta sobre *Attention Is All You Need* y *Retrieval-Augmented Generation*. "
                "El desplegable filtra la recuperación en Qdrant por `metadata.source`.")
    with gr.Row():
        pregunta_in = gr.Textbox(label="Pregunta", lines=2, scale=4,
                                 placeholder="Por ejemplo: What is the Transformer?")
        paper_in = gr.Dropdown(choices=list(FUENTES), value="Todos", label="Paper", scale=1)
    boton = gr.Button("Preguntar", variant="primary")
    respuesta_out = gr.Markdown(label="Respuesta", elem_id="respuesta")
    fuentes_out = gr.Dataframe(headers=["Título", "Archivo", "Score"], label="Fuentes recuperadas",
                               interactive=False, elem_id="fuentes")
    boton.click(ui_responder, [pregunta_in, paper_in], [respuesta_out, fuentes_out])
    pregunta_in.submit(ui_responder, [pregunta_in, paper_in], [respuesta_out, fuentes_out])

PUERTO = 7861
demo.launch(server_name="127.0.0.1", server_port=PUERTO, prevent_thread_lock=True, quiet=True)
print("App Gradio en http://127.0.0.1:%d" % PUERTO)

In [ ]:
import subprocess, sys, tempfile, textwrap

EVID = Path("../evidencias") if Path("../evidencias").exists() else Path("evidencias")
SCRIPT_CAPTURA = textwrap.dedent('''
    import sys
    from playwright.sync_api import sync_playwright
    url, pregunta, paper, salida = sys.argv[1:5]
    with sync_playwright() as p:
        b = p.chromium.launch()
        pg = b.new_page(viewport={"width": 1280, "height": 900}, device_scale_factor=1.5)
        pg.goto(url, wait_until="networkidle")
        pg.get_by_label("Pregunta").fill(pregunta)
        pg.get_by_label("Paper").click()
        pg.get_by_role("option", name=paper).click()
        pg.get_by_role("button", name="Preguntar").click()
        pg.wait_for_function("document.querySelector('#respuesta') && document.querySelector('#respuesta').innerText.trim().length > 40", timeout=240000)
        pg.wait_for_timeout(1500)
        pg.screenshot(path=salida, full_page=True)
        b.close()
''')
script_path = Path(tempfile.gettempdir()) / "captura_gradio.py"
script_path.write_text(SCRIPT_CAPTURA)

salida_png = EVID / "C3_gradio.png"
r = subprocess.run([sys.executable, str(script_path), f"http://127.0.0.1:{PUERTO}", PREGUNTA_COMPARA,
                    "Retrieval-Augmented Generation", str(salida_png)],
                   capture_output=True, text=True, timeout=400)
print("captura:", "ok" if r.returncode == 0 else r.stderr[-500:], "|", salida_png, salida_png.exists())

In [ ]:
# Cierra el servidor al terminar y comprueba que el puerto quedó libre.
demo.close()
import socket
with socket.socket() as s:
    s.settimeout(1)
    libre = s.connect_ex(("127.0.0.1", PUERTO)) != 0
print("Servidor Gradio cerrado; puerto libre:", libre)

## C6 · Persistencia del índice en disco

Cambio mínimo: `QdrantClient(":memory:")` pasa a `QdrantClient(path=...)`. Para que "reiniciar el runtime" sea real, la reapertura la hago en **un proceso Python nuevo** (`subprocess`), que no comparte memoria con este kernel. Mido (1) indexar desde cero y (2) reabrir la colección ya existente y consultarla sin reindexar.

In [ ]:
import json, shutil

RUTA_QDRANT = Path("./qdrant_data/c6")      # qdrant_data/ ya está en .gitignore
COLECCION_C6 = "academic_papers_nvidia"
shutil.rmtree(RUTA_QDRANT.parent, ignore_errors=True)   # empezar desde cero para medir de verdad

docs_c6 = documentos_para(900)
t0 = time.perf_counter()
cliente_disco = QdrantClient(path=str(RUTA_QDRANT))
if cliente_disco.collection_exists(COLECCION_C6):
    print("La colección ya existía: se reutiliza")
else:
    vs_disco = crear_vectorstore(cliente_disco, COLECCION_C6, docs_c6)
t_indexar = time.perf_counter() - t0
n_puntos = cliente_disco.count(COLECCION_C6).count
cliente_disco.close()          # libera el candado del directorio antes de que lo abra otro proceso
tam_mb = sum(f.stat().st_size for f in RUTA_QDRANT.rglob("*") if f.is_file()) / 1e6
print(f"Indexación desde cero: {t_indexar:.1f} s, {n_puntos} puntos, {tam_mb:.1f} MB en disco")

In [ ]:
SCRIPT_REABRIR = textwrap.dedent('''
    import json, os, sys, time
    t_ini = time.perf_counter()
    from dotenv import find_dotenv, load_dotenv
    load_dotenv(find_dotenv(usecwd=True))
    from qdrant_client import QdrantClient
    from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
    from langchain_qdrant import QdrantVectorStore
    t_imports = time.perf_counter() - t_ini
    ruta, coleccion, pregunta = sys.argv[1:4]
    t0 = time.perf_counter()
    client = QdrantClient(path=ruta)
    existe = client.collection_exists(coleccion)
    n = client.count(coleccion).count if existe else 0
    t_abrir = time.perf_counter() - t0
    emb = NVIDIAEmbeddings(model="nvidia/nemotron-3-embed-1b", truncate="END")
    vs = QdrantVectorStore(client=client, collection_name=coleccion, embedding=emb)
    t0 = time.perf_counter()
    docs = vs.similarity_search(pregunta, k=3)
    t_consulta = time.perf_counter() - t0
    print(json.dumps(dict(existe=existe, puntos=n, abrir_s=t_abrir, consulta_s=t_consulta, imports_s=t_imports,
                          top=[(d.metadata["source"], d.page_content[:90].replace(chr(10), " ")) for d in docs])))
    client.close()
''')
ruta_script = Path(tempfile.gettempdir()) / "reabrir_qdrant.py"
ruta_script.write_text(SCRIPT_REABRIR)

mediciones = []
for intento in range(3):
    r = subprocess.run([sys.executable, str(ruta_script), str(RUTA_QDRANT), COLECCION_C6,
                        "How many attention heads does the base Transformer use?"],
                       capture_output=True, text=True, timeout=300)
    if r.returncode != 0:
        print(r.stderr[-600:]); break
    mediciones.append(json.loads(r.stdout.strip().splitlines()[-1]))
for m in mediciones:
    print({k_: (round(v, 3) if isinstance(v, float) else v) for k_, v in m.items() if k_ != "top"})
print("Top-3 tras reabrir sin reindexar:")
for fuente, texto in mediciones[0]["top"]:
    print("  ", fuente, "|", texto)

abrir_med = float(np.median([m["abrir_s"] for m in mediciones]))
consulta_med = float(np.median([m["consulta_s"] for m in mediciones]))
df_c6 = pd.DataFrame([
    dict(operacion="Indexar desde cero (embeddings + inserción)", segundos=round(t_indexar, 2)),
    dict(operacion="Reabrir colección en proceso nuevo (sin reindexar)", segundos=round(abrir_med, 3)),
    dict(operacion="Primera consulta tras reabrir (incluye embedding de la pregunta)", segundos=round(consulta_med, 3)),
])
df_c6["veces_mas_rapido_que_indexar"] = (t_indexar / df_c6["segundos"]).round(1)
df_c6

## C2 · Un tercer paper: el RAG que explica HNSW

Agrego *Efficient and robust approximate nearest neighbor search using Hierarchical Navigable Small World graphs* (Malkov y Yashunin, arXiv 1603.09320) al diccionario `PAPERS`, repito conversión, chunking (900) e indexación en una colección nueva y evalúo 4 preguntas con ground truth. **Verifiqué cada palabra clave leyendo el texto extraído del PDF** (la celda las busca en el texto completo antes de evaluar). El PDF de HNSW está en dos columnas, así que MarkItDown intercala columnas en algunos pasajes: es parte de lo que hay que tener en cuenta al leer los resultados.

In [ ]:
PAPERS["hnsw_paper.pdf"] = "https://arxiv.org/pdf/1603.09320"
TITULOS["hnsw_paper.pdf"] = "Hierarchical Navigable Small World graphs"
download_file(PAPERS["hnsw_paper.pdf"], DOCS_DIR / "hnsw_paper.pdf")
TEXTOS["hnsw_paper.pdf"] = md_converter.convert(str(DOCS_DIR / "hnsw_paper.pdf")).text_content
print(f"hnsw_paper.pdf: {len(TEXTOS['hnsw_paper.pdf']):,} caracteres")

docs3 = documentos_para(900)
por_fuente = {s: sum(d.metadata["source"] == s for d in docs3) for s in PAPERS}
print("Chunks por paper:", por_fuente, "| total:", len(docs3))
t0 = time.perf_counter()
vs3 = crear_vectorstore(qdrant_client, "papers3_cs900", docs3)
print(f"Indexado en {time.perf_counter() - t0:.1f} s")

In [ ]:
# Cada pregunta: lista de pares (fuente, palabra_clave) que deben aparecer entre los chunks recuperados.
preguntas_c2 = [
    dict(id="H1", k=5, pregunta="How is the maximum layer of a new element selected in HNSW?",
         verdad=[("hnsw_paper.pdf", "exponentially decaying")],
         esperado="Se sortea con una distribución de probabilidad que decae exponencialmente (l = floor(-ln(unif(0..1))·mL))."),
    dict(id="H2", k=5, pregunta="What is a reasonable range for the parameter M in HNSW?",
         verdad=[("hnsw_paper.pdf", "5 to 48")],
         esperado="Un rango razonable de M es de 5 a 48."),
    dict(id="H3", k=5, pregunta="What is a simple choice for the level normalization parameter mL?",
         verdad=[("hnsw_paper.pdf", "1/ln(M)")],
         esperado="mL = 1/ln(M), equivalente a p = 1/M en una skip list."),
    dict(id="H4 (cruce)", k=6, pregunta="The RAG paper builds its index with FAISS and an HNSW approximation. What is that HNSW approximation and how does it speed up retrieval?",
         verdad=[("rag_paper.pdf", "FAISS"), ("hnsw_paper.pdf", "Navigable Small World")],
         esperado="RAG indexa 21M de documentos con FAISS y HNSW; HNSW es un grafo multicapa navegable que da búsqueda aproximada de complejidad logarítmica."),
]
for p in preguntas_c2:
    for fuente, clave in p["verdad"]:
        assert clave.lower() in TEXTOS[fuente].lower(), (p["id"], fuente, clave)
print("Ground truth verificado contra el texto de los PDFs: todas las claves aparecen.")

filas_c2 = []
for p in preguntas_c2:
    docs = con_reintentos(vs3.as_retriever(search_kwargs={"k": p["k"]}).invoke, p["pregunta"])
    aciertos = [any(clave.lower() in d.page_content.lower() and d.metadata["source"] == fuente for d in docs)
                for fuente, clave in p["verdad"]]
    respuesta = con_reintentos(cadena_respuesta.invoke, {"context": format_docs(docs), "question": p["pregunta"]}).strip()
    filas_c2.append(dict(id=p["id"], fuentes=[f"{d.metadata['source'].split('_')[0]}#{d.metadata['chunk_index']}" for d in docs],
                         hit=all(aciertos), respuesta=respuesta, esperado=p["esperado"]))
    print("=" * 100)
    print(p["id"], "|", p["pregunta"])
    print("Fuentes recuperadas:", filas_c2[-1]["fuentes"], "| acierto del ground truth:", all(aciertos))
    print("Respuesta del RAG:", respuesta)
    print("Esperado:", p["esperado"])